# Automatic Text Summarization


**Goal:** Generate a concise summary of the texts.

**How:** Unsupervised text summarization (extractive summarization) using TextRank Alg.

**Data:** Manually Annotated Sub-Corpus First Release.


In [ ]:
# importing packages & libraries
import numpy as np
import pandas as pd
import nltk
import re
import networkx as nx

from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize
from sklearn.metrics.pairwise import cosine_similarity

# nltk.download('punkt')
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', -1)
pd.set_option('max_colwidth', 800)

In [ ]:
! pip install networkx

### Data

In [ ]:
# reading data
df = pd.read_csv('./data/all_data_new.csv')
df = df[['file_name','text']]

In [ ]:
df.head(2)

### Getting setences

In [ ]:
# splitting text into sentences
sentences = []
for s in df['text']:
    sentences.append(sent_tokenize(s))

sentences = [y for x in sentences for y in x] 

In [ ]:
# printing 2 sentences
sentences[:2]

### Word Embeddings

**Good:** GloVe keeps word order. 

**Not so good:** Size of 822 MB.

In [ ]:
!wget http://nlp.stanford.edu/data/glove.6B.zip

In [ ]:
!unzip glove*.zip

In [ ]:
# extracting word vectors
word_embeddings = {}
f = open('glove.6B.100d.txt', encoding='utf-8')
for line in f:
    values = line.split()
    word = values[0]
    coefs = np.asarray(values[1:], dtype='float32')
    word_embeddings[word] = coefs
f.close()

In [ ]:
# getting the lengh of word embeddings
len(word_embeddings)

In [ ]:
# removing punctuations, numbers and special characters
clean_sentences = pd.Series(sentences).str.replace("[^a-zA-Z]", " ")

# making alphabets lowercase
clean_sentences = [s.lower() for s in clean_sentences]

In [ ]:
# defining stop words
stop_words = stopwords.words('english')

In [ ]:
# removing stopwords
def remove_stopwords(sen):
    sen_new = " ".join([i for i in sen if i not in stop_words])
    return sen_new

In [ ]:
# removing stopwords from the sentences
clean_sentences = [remove_stopwords(r.split()) for r in clean_sentences]

### Vectors

In [ ]:
# extracting word vectors
word_embeddings = {}
f = open('glove.6B.100d.txt', encoding='utf-8')
for line in f:
    values = line.split()
    word = values[0]
    coefs = np.asarray(values[1:], dtype='float32')
    word_embeddings[word] = coefs
f.close()

In [ ]:
# creating vectors for sentences
sentence_vectors = []
for i in clean_sentences:
    if len(i) != 0:
        v = sum([word_embeddings.get(w, np.zeros((100,))) for w in i.split()])/(len(i.split())+0.001)
    else:
        v = np.zeros((100,))
    sentence_vectors.append(v)

### Similarity Matrix

In [ ]:
# defining a similarity matrix
sim_mat = np.zeros([len(sentences), len(sentences)])

In [ ]:
# initializing similarity matrix
for i in range(len(sentences)):
    for j in range(len(sentences)):
        if i != j:
            sim_mat[i][j] = cosine_similarity(sentence_vectors[i].reshape(1,100), sentence_vectors[j].reshape(1,100))[0,0]

### Text Rank Alg

In [ ]:
# converting similarity matrix to a graph
nx_graph = nx.from_numpy_array(sim_mat)
scores = nx.pagerank(nx_graph)
# scores = nx.textrank(nx_graph)

In [ ]:
# defining ranked sentences
ranked_sentences = sorted(((scores[i],s) for i,s in enumerate(sentences)), reverse=True)

In [ ]:
# extracting top 10 sentences as the summary
for i in range(10):
    print(ranked_sentences[i][1])

In [ ]:
ranked_sentences

In [ ]:
len(ranked_sentences)